In [1]:
!pip install nltk
!pip install spacy
!pip install pyarrow
!pip install --upgrade pandas pyarrow


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import spacy
from collections import Counter
import os

# 1. Setup paths for both files
splits_dir = os.path.join('..', 'data', 'splits')
files_to_process = {
    'train': os.path.join(splits_dir, 'train.parquet'),#manhattan
    'test': os.path.join(splits_dir, 'test.parquet')#philadelphia
}

# 2. Setup spaCy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

def extract_complex_landmarks(text_list):
    counts = Counter()
    blacklist = {
        'block', 'turn', 'way', 'left', 'right', 
        'ave', 'meters', 'feet', 'side', 'corner', 'end', 'direction',
        'south','southeast','southwest',
        'west','east','north','northeast','northwest'
    }

    print(f"Processing {len(text_list)} instructions...")
    # We enable the 'attribute_ruler' and 'lemmatizer' for better noun phrase detection
    for doc in nlp.pipe(text_list.astype(str), batch_size=500, disable=["ner"]):
        
        # --- Strategy: Use noun_chunks to find multi-word landmarks ---
        # This catches "Dunkin Donuts", "Central Park", "Small Cafe", etc.
        for chunk in doc.noun_chunks:
            # Clean the chunk text
            clean_words = [t.text.lower() for t in chunk if t.text.lower() not in blacklist and not t.is_stop and len(t.text) > 3]
            #clean_words = [t.text.lower() for t in chunk if t.text.lower() not in blacklist and not t.is_stop and len(t.text) > 2]
            
            if clean_words:
                phrase = " ".join(clean_words)
                counts.update([phrase])

    return counts

# --- EXECUTION ---
all_results = []

for split_name, path in files_to_process.items():
    if os.path.exists(path):
        print(f"✅ Loading {split_name} data...")
        df = pd.read_parquet(path, engine='pyarrow')
        
        # Extract landmarks (including multi-word phrases)
        landmark_counts = extract_complex_landmarks(df['content'])
        
        # Convert to list for DataFrame
        for landmark, freq in landmark_counts.most_common(100):
            all_results.append({
                'Split': split_name,
                'Landmark': landmark.upper(),
                'Frequency': freq
            })
    else:
        print(f"❌ File not found: {path}")

# 3. Save to File
if all_results:
    results_df = pd.DataFrame(all_results)
    output_file = 'frequent_landmarks_report.csv'
    results_df.to_csv(output_file, index=False)
    print(f"\n📁 SUCCESS: Frequent landmarks saved to '{output_file}'")

    # Display Top Multi-word results for the Train set
    print("\n--- Top Multi-Word Landmarks Found ---")
    print(results_df[results_df['Landmark'].str.contains(' ')].head(20))

✅ Loading train data...
Processing 7000 instructions...
✅ Loading test data...
Processing 1278 instructions...

📁 SUCCESS: Frequent landmarks saved to 'frequent_landmarks_report.csv'

--- Top Multi-Word Landmarks Found ---
    Split              Landmark  Frequency
1   train       BICYCLE PARKING       1188
28  train  FAST FOOD RESTAURANT        201
30  train        BICYCLE RENTAL        174
32  train           POST OFFICE        164
41  train  DUANE READE PHARMACY        136
42  train        DRINKING WATER        135
47  train            CHASE BANK        118
48  train      LEXINGTON AVENUE        117
50  train          CLOTHES SHOP        115
52  train     HISTORIC MEMORIAL        107
54  train           14TH STREET        102
57  train          BIKE PARKING         96
59  train     HISTORIC DISTRICT         89
60  train          ALCOHOL SHOP         87
61  train      PARKING ENTRANCE         85
62  train           VACANT SHOP         85
63  train     HISTORIC BUILDING         84
64 